# Part 1: Analysis workflow

In this exercise we are going to use a dataset from the CMS Open Data portal. It is a DoubleMuon primary dataset in NANOAOD format from RunG of 2016. Run period from run number 278820 to 280385. [1] It contains muon data from events from CMS collisions with at least 2 muons.

NANOAOD datasets are in the ROOT tree format. This is a flat data structure, where we have for each collision (event) a list of muon particles that were reconstructed from the recorded detector signals. For each particle, we have a list of measured variables. Each of this variables is a "brnch" of the ROOT tree. 

[1] https://opendata.cern.ch/record/30522

In [ ]:
cd ~/work/CmsOpenData/RDFAnalyzer; pwd

### 1. Import ROOT and necessary tools

In [ ]:
# Import ROOT (ROOT is a analysis package used to handle 
# the tree files, but also histograms, etc.)
import ROOT as R
%jsroot off
from utils import histpars, loadRDF, plot

In [ ]:
R.gROOT.ProcessLine('.L ./_tdrstyle.C')
R.setTDRStyle()
R.gROOT.SetBatch(1)

### 2. Load dataset

To load and manipulate the data, we are going to use the ROOT RDataFrame structure [2]. ROOT's RDataFrame offers a modern, high-level interface for analysis of data stored in TTree , CSV and other data formats, in C++ or Python.

[2] https://root.cern/doc/v628/classROOT_1_1RDataFrame.html

In [ ]:
# Load the data into an RDataFrame structure
datafile = "../data_skimmed/Run2016G_MET_NANOAOD_UL2016_MiniAODv2_NanoAODv9-v1_270000_6A4F07DD-F1D1-164F-B509-AFBA9877D6D5_skimmed.root"
df = loadRDF("Events", datafile)

### 3. Fill the histograms with raw data

In this first stage, we want to look at the distribution of some key variables of the muons. The goal is to get familiar with the muon properties and to design the selection that we will apply later.

Below you have a list of the histograms that we are going to produce.

In [ ]:
print("\nList of histograms to be produced:\n")
for var in histpars:
    print(f"{var:18} : {histpars[var]['name']:40}")

The produced histograms will be saved in a .root file, so that we can plot them later without having to recompute them.

In [ ]:
# Make all histograms
hists = {}
for variable in histpars:
    _xlabel = histpars[variable]["xlabel"]
    _bins   = histpars[variable]["bins"]
    _xmin   = histpars[variable]["xmin"]
    _xmax   = histpars[variable]["xmax"]
    h = df.Histo1D((f"h_{variable}",f";{_xlabel};Events", _bins, _xmin, _xmax), variable)
    hists[variable] = h

print("Writing to file...")
fwrite = R.TFile.Open("temp/hists_raw.root","RECREATE")
for variable in histpars:
    hists[variable].Write()
fwrite.Close()

print("Done!")

In [ ]:
# Insert variable to read
variable = "DiMuon_invMass"

fread = R.TFile.Open("temp/hists_raw.root","READ")
h = fread.Get(f"h_{variable}")
h.SetDirectory(0)
fread.Close()

R.gROOT.FindObject(f"c_{variable}") and R.gROOT.FindObject(f"c_{variable}").Close() # Avoid memory conflicts and crashes
c = R.TCanvas(f"c_{variable}","")
c = plot(c, h)
c.Draw()

### 4. Signal vs background discrimination

Now we will apply a selection to remove the background and isolate Z to mu mu events. This selection consists in a series of cuts on the muons' variables

In [ ]:
# Import ROOT (ROOT is a analysis package used to handle 
# the tree files, but also histograms, etc.)
import ROOT as R
%jsroot off
from utils import histpars, loadRDF, plot

In [ ]:
# Selection parameters
isGlobal   = 0
isTight    = 0
pt_min     = 0.
abseta_max = 2.4
dz_max     = 0.4 # suggested: [0.02, 0.2, 0.4]
dxy_max    = 0.2 # suggested: [0.02, 0.1, 0.2]
relIso_max = 5. # Try different values

cuts = {
    'GLB' : f'Muon_isGlobal=={isGlobal}',
    'TGT' : f'Muon_tightId=={isTight}',
    'PT'  : f'Muon_pt>{pt_min}',
    'ETA' : f'abs(Muon_eta)<{abseta_max}',
    'DZ'  : f'Muon_dz<{dz_max}',
    'DXY' : f'Muon_dxy<{dxy_max}',
    'ISO' : f'Muon_tkRelIso<{relIso_max}'
}

cutstring = '&&'.join([f"({cuts[cut]})" for cut in cuts])
print(cutstring)

In [ ]:
df_cut = loadRDF("Events", datafile)
df_cut = df_cut.Redefine("Muon_isGoodMuon",f"return RVecI({cutstring})")
df_cut = df_cut.Redefine("DiMuon_invMass","invariantMass(Muon_isGoodMuon, Muon_pt, Muon_eta, Muon_phi)")
df_cut = df_cut.Filter("DiMuon_invMass>=0.")
for variable in histpars:
    if variable.startswith("Muon_"):
        df_cut = df_cut.Redefine(variable, f"({variable})[Muon_isGoodMuon]")

In [ ]:
# Make all histograms
hists_sel = {}
for variable in histpars:
    if variable not in df_cut.GetColumnNames():
        print(f"Warning: {variable} not in DataFrame")
        continue
    _xlabel = histpars[variable]["xlabel"]
    _bins   = histpars[variable]["bins"]
    _xmin   = histpars[variable]["xmin"]
    _xmax   = histpars[variable]["xmax"]
    h = df_cut.Histo1D((f"h_{variable}_sel",f";{_xlabel};Events", _bins, _xmin, _xmax), variable)
    hists_sel[variable] = h

print("Writing to file...")
fwrite_sel = R.TFile.Open("temp/hists_selection.root","RECREATE")
for variable in histpars:
    hists_sel[variable].Write()
fwrite_sel.Close()

print("Done!")

In [ ]:
# Insert variable to read
variable = "DiMuon_invMass"

fread = R.TFile.Open("temp/hists_raw.root","READ")
h_raw = fread.Get(f"h_{variable}")
h_raw.SetDirectory(0)
fread.Close()

fread = R.TFile.Open("temp/hists_selection.root","READ")
h_sel = fread.Get(f"h_{variable}_sel")
h_sel.SetDirectory(0)
fread.Close()

R.gROOT.FindObject(f"c_{variable}") and R.gROOT.FindObject(f"c_{variable}").Close() # Avoid memory conflicts and crashes
c = R.TCanvas(f"c_{variable}","")
c = plot(c, [h_raw, h_sel], logy=True)
c.Draw()